# LAT 1 transition: VAE with pairwise distances


In [1]:
from deep_cartograph.tools import traj_projection, compute_features
from deep_cartograph.deep_carto import deep_cartograph 
from deep_cartograph.modules.common import check_data
import importlib.resources as resources
from deep_cartograph import data

from IPython.display import display, HTML
from typing import List, Optional
import matplotlib.pyplot as plt
from decimal import Decimal
from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import logging
import yaml
import time
import os

# Get the path to the data
data_folder = resources.files(data)

# Set logging level
logging.basicConfig(level=logging.INFO)

def project_evaluation_data(evaluation_traj_data: str,
                            evaluation_top_data: str,
                            output_folder,
                            model_name) -> List[str]:
    """ 
    Project evaluation data and return the paths to the projected data files.
    """
    
    trajectories, topologies = check_data(evaluation_traj_data, evaluation_top_data)
    trajectory_names = [Path(traj).stem for traj in trajectories]
    
    # Read configuration used to compute features
    compute_features_config = os.path.join(output_folder, 'compute_features', 'configuration.yml')
    with open(compute_features_config) as config_file:
        configuration = yaml.load(config_file, Loader = yaml.FullLoader)
    reference_topology = os.path.join(output_folder, 'compute_features', 'ref_topology.pdb')

    # Compute features
    args = {
        'configuration': configuration, 
        'trajectories': trajectories, 
        'topologies': topologies, 
        'reference_topology': reference_topology,
        'output_folder': os.path.join(output_folder, 'compute_features_eval')
    }
    traj_colvars_paths = compute_features(**args)
    
    # Read configuration to project trajectories
    projection_config = os.path.join(output_folder, 'traj_projection', 'configuration.yml')
    with open(projection_config) as config_file:
        configuration = yaml.load(config_file, Loader = yaml.FullLoader)
    
    # Find model path
    model_path = os.path.join(output_folder, 'train_colvars', model_name, 'model.zip')
    
    # Project evaluation trajectories
    args = { 
        'configuration' : configuration,
        'colvars_paths': traj_colvars_paths,
        'topologies': topologies,
        'trajectory_names': trajectory_names,
        'model_paths': [model_path],
        'model_traj_paths': None,
        'output_folder': os.path.join(output_folder, 'traj_projection_eval')
    }
    proj_eval_data = traj_projection(**args)
    
    return proj_eval_data.get(model_name, {}).get('traj_paths', [])

def create_time_evolution(proj_training_data_path: str, 
                          proj_evaluation_data_paths: List[str], 
                          output_path: str):
    """
    Create a time evolution plot with the evaluation data represented as a 
    min-max shadow (error/generalization spread) behind the training data.
    """

    # Read projected training data
    proj_data = pd.read_csv(proj_training_data_path)
    
    proj_eval_data = []
    if len(proj_evaluation_data_paths) > 0:
        logging.info(f"Processing {len(proj_evaluation_data_paths)} evaluation datasets for error estimation.")
        # Read projected evaluation data
        for eval_path in proj_evaluation_data_paths:
            proj_eval_data.append(pd.read_csv(eval_path))

    # Create time arrays
    # We do not assume that the evaluation data has the same time steps as the training data.
    train_time_array = np.arange(len(proj_data))
    eval_time_array = np.linspace(0, len(proj_data)-1, num=len(proj_eval_data[0])) if len(proj_eval_data) > 0 else None
    
    # Find number of components
    n_components = proj_data.shape[1]

    # --- Custom Font Sizes ---
    LABEL_SIZE = 18
    TITLE_SIZE = 18
    TICK_SIZE = 18
    LEGEND_SIZE = 18

    # Create plot
    plt.figure(figsize=(10, 8))
    
    # Use a colormap to ensure the line and its shadow share the same color
    colors = plt.cm.get_cmap('tab10', n_components)

    for i in range(n_components):
        # Get specific color for this CV
        c = colors(i)
        
        # 1. Plot the Training Data (The main signal)
        plt.plot(train_time_array, proj_data.iloc[:, i], 
                 label=f'Training data - CV {i+1}', 
                 color=c, 
                 linewidth=2,
                 zorder=2) # zorder=2 ensures line is on top of shadow
        
        # 2. Calculate and Plot the Shadow (The evaluation spread)
        if len(proj_evaluation_data_paths) > 0:
            # Extract the i-th column from all evaluation dataframes
            # resulting list shape: [ (N_steps,), (N_steps,), ... ]
            eval_values_list = [df.iloc[:, i].values for df in proj_eval_data]
            
            # Stack them to get a shape of (N_eval_datasets, N_steps)
            # This allows us to compute stats column-wise
            eval_matrix = np.vstack(eval_values_list)
            
            # Calculate Min and Max across the datasets (axis=0)
            eval_min = np.min(eval_matrix, axis=0)
            eval_max = np.max(eval_matrix, axis=0)
            
            # Create the shadow
            # We use the same 'train_time_array' for x-axis as per "training + noise" assumption
            plt.fill_between(eval_time_array, eval_min, eval_max, 
                             color=c, 
                             alpha=0.3, # Transparency
                             label=f'Evaluation data - CV {i+1}',
                             zorder=1) # zorder=1 ensures shadow is behind line

    # Apply Font Sizes
    plt.xlabel('Time step', fontsize=LABEL_SIZE)
    plt.ylabel('Projected CV', fontsize=LABEL_SIZE)
    plt.title('Time Evolution: Generalization Capability', fontsize=TITLE_SIZE)
    
    # Apply tick label sizes
    plt.xticks(fontsize=TICK_SIZE)
    plt.yticks(fontsize=TICK_SIZE)
    
    # Apply legend size
    plt.legend(fontsize=LEGEND_SIZE)
    
    plt.grid(True, alpha=0.5)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

def show_results(output_folder: str, 
                 model_name: str, 
                 system: str, 
                 evaluation_traj_data: Optional[str] = None, 
                 evaluation_top_data: Optional[str] = None):
    """
    Show the results for a specific model trained with deep cartograph

    Inputs
    ------

        output_folder   (str):          path to the output folder
        model_name      (str):          name of the model
    """

    def show_score(score_path):
        """
        Print score path in a nice format 

        Input
        -----

            score_path: path to the score file
        """

        # Read score
        with open(score_path, 'r') as file:
            score = file.read()

        # Print score in scientific notation
        print(f"Final model score: {Decimal(score):.4E}")

    def show_eigenvalues(eig_path):
        """
        Print eigenvalues in a nice format

        Input
        -----

            eig_path: path to the eigenvalues file
        """

        # Read eigenvalues
        with open(eig_path, 'r') as file:
            eigenvalues = file.readlines()

        # Print eigenvalues
        for i, eig in enumerate(eigenvalues):
            print(f"Eigenvalue {i+1}: {Decimal(eig):.4E}")

    # Check necessary folders
    print(f"Output folder: {output_folder}")
    train_cv_folder = os.path.join(output_folder, 'train_colvars')
    if not os.path.exists(train_cv_folder):
        print("Training folder not found")
        return
    traj_projection_folder = os.path.join(output_folder, 'traj_projection')
    if not os.path.exists(traj_projection_folder):
        print("Trajectory projection folder not found")
        return
    model_folder = os.path.join(train_cv_folder, model_name)
    if not os.path.exists(model_folder):
        print("Model folder not found")
        return
    training_folder = os.path.join(model_folder, 'training')
    if not os.path.exists(training_folder):
        print("Training folder not found")
        return
    traj_data_folder = os.path.join(model_folder, 'traj_data', system)
    if not os.path.exists(traj_data_folder):
        print("Trajectory data folder not found")
        return
    sensitivity_folder = os.path.join(model_folder, 'sensitivity_analysis')
    if not os.path.exists(sensitivity_folder):
        print("Sensitivity analysis folder not found")
        return

    # Show score and eigenfunctions if any
    if model_name in ['vae', 'ae', 'deep_tica']:
        score_path = os.path.join(training_folder, 'model_score.txt')
        if os.path.exists(score_path):
            show_score(score_path)
        else: print("Warning: Score file not found")
    if model_name == 'deep_tica':
        eig_path = os.path.join(training_folder, 'eigenvalues.txt')
        if os.path.exists(eig_path):
            show_eigenvalues(eig_path)
        else: print("Warning: Eigenvalues file not found")

    # Show evolution of CV for training data and given evaluation data
    if evaluation_traj_data is not None and evaluation_top_data is not None:
        proj_evaluation_data_paths = project_evaluation_data(evaluation_traj_data,
                                                            evaluation_top_data,
                                                            output_folder,
                                                            model_name)
    else:
        proj_evaluation_data_paths = []
    proj_training_data_path = os.path.join(traj_data_folder, 'projected_trajectory.csv')
    proj_train_plot = os.path.join(traj_data_folder, 'projected_trajectory.png')
    if os.path.exists(proj_training_data_path):
        create_time_evolution(proj_training_data_path, 
                              proj_evaluation_data_paths,
                              proj_train_plot)
    
    # Paths to images
    sensitivity_histogram = os.path.join(sensitivity_folder, 'sensitivity_histogram.png')
    trajectory = os.path.join(traj_data_folder, 'trajectory.png')
    loss = os.path.join(training_folder, 'loss.png')
    beta_loss = os.path.join(training_folder, 'vae_kl_reconstruction_loss.png')
    beta_evolution = os.path.join(training_folder, 'vae_beta.png')
    eigenvalues = os.path.join(training_folder, 'eigenvalues.png')
    fes = os.path.join(traj_projection_folder, model_name, 'fes', f'fes_{model_name}_1', 'fes.png')
    paths = [trajectory, loss, beta_loss, eigenvalues, proj_train_plot, fes, sensitivity_histogram, beta_evolution]

    # Generate HTML image tags
    timestamp = int(time.time())
    images_html = [f'<img src="{path}?{timestamp}" style="width: 600px; margin-right: 10px;">' for path in paths if os.path.exists(path)]
    if not images_html:
        print("Warning: No images to display.")
        return

    # Display images
    display(HTML(''.join(images_html)))

/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


Some examples from previous trainings:

/shared/work/pnavarro/projects/2025_DeepCartograph/LAT1/chain_B/deepCarto_analysis_iteration_4_dist_smallFeat_smallVAE_optimization/output_start_beta_0.000000001

/shared/work/pnavarro/projects/2025_DeepCartograph/LAT1/chain_B/deepCarto_analysis_iteration_4_dist_smallFeat_largeVAE_optimization/output_weight_decay_0.000000001



## Project all GOdMD trajectories

In [ ]:
from deep_cartograph.deep_carto import deep_cartograph 
import importlib.resources as resources
from deep_cartograph import data
import shutil
import yaml
import os

data_folder = resources.files(data)

features = "distances"
config_folder = f"{data_folder}/lat1/input"
config_path = f"{config_folder}/{features}_config_godmds.yml"
cv_models = ['vae']

# Main trajectory and topology  ---> to give as main (no augmentation) or seed (augmentation)
version = "GOdMD_v1"
training_replica = f"_raw_godmds"
input_path = f"{data_folder}/lat1/input"
traj_data = os.path.join(input_path, 'training_data_godmds', 'trajs') 
top_data = os.path.join(input_path, 'training_data_godmds', 'tops')

# (Optional) Endpoint equilibrations ---> supplementary data to project onto the CV
# Restrained equilibrations at the endpoints of the reaction coordinate
sup_input_path = f"{data_folder}/lat1/input/sup_data"
sup_traj_data = os.path.join(sup_input_path, 'trajs')
sup_top_data = os.path.join(sup_input_path, 'tops')

waypoints_data = f"{data_folder}/lat1/input/waypoints_data"
with open(config_path) as config_file:
    configuration = yaml.load(config_file, Loader = yaml.FullLoader)
    
configuration['train_colvars']['common']['training']['kl_annealing']['max_beta'] = 0.005
    
# Output folder for the full workflow
output_folder = f"{data_folder}/lat1/output/{version}/tests/{features}{training_replica}"

# Run workflow
deep_cartograph(
    configuration = configuration,
    trajectory_data = traj_data,
    topology_data = top_data,
    supplementary_traj_data = sup_traj_data,
    supplementary_top_data = sup_top_data,
    waypoints_data = sup_top_data,
    cvs = cv_models,
    dimension = 2,
    restart = True,
    output_folder = output_folder)


/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(
/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "
/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:777: UserWarning: Unit cell

KLAAnnealing initialized with type=sigmoid, start_beta=1e-07, max_beta=0.001)
LROnPlateauManager initialized. Will start monitoring validation loss at epoch 1625.


/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "
/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:777: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn("Unit cell dimensions not found. "
/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no information for attr: 'elements' Using default value of ' '
  warnings.warn("Found no information for attr: '{}'"
/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no informatio

KEY:  data


Plotting only the first 20 features out of 55.
Plotting only the first 20 features out of 55.


/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "
/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:777: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn("Unit cell dimensions not found. "
/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no information for attr: 'elements' Using default value of ' '
  warnings.warn("Found no information for attr: '{}'"
/home/pnavarro/.conda/envs/deep_cartograph/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no informatio

...

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>

<Figure size 500x400 with 0 Axes>